# 04. Генерация эмбеддингов (IBM Granite)

**Модель:** [`ibm-granite/granite-embedding-311m-multilingual-r2`](https://huggingface.co/ibm-granite/granite-embedding-311m-multilingual-r2)
- 311M параметров, 768-dim эмбеддинги, L2-normalized
- Макс. контекст: 8192 токена, 12+ языков

**План:**
1. Загрузить корпус `df_corpus_ready.pkl`
2. Разбить документы на чанки по 512 токенов (overlap 64)
3. Батчевая генерация эмбеддингов
4. Mean pooling + L2-нормализация
5. Сохранить `embeddings.npy` + `chunks_meta.pkl`

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

OUT_DIR = Path('artifacts')
OUT_DIR.mkdir(exist_ok=True)

MODEL_ID    = 'ibm-granite/granite-embedding-311m-multilingual-r2'
MAX_LEN     = 512
OVERLAP     = 64
BATCH_SIZE  = 16
TEXT_COL    = 'clean_text'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device      : {device}')
if device.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM total  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

device      : cuda
GPU         : NVIDIA GeForce RTX 4060 Laptop GPU
VRAM total  : 8.6 GB


In [2]:
df = pd.read_pickle(OUT_DIR / 'df_corpus_ready.pkl').reset_index(drop=True)
print(f'Документов: {len(df)}')
print(f'Колонки   : {list(df.columns)}')

assert TEXT_COL in df.columns, f'{TEXT_COL!r} нет в корпусе'
print(f'\nПример {TEXT_COL} (первые 200 симв):')
print(df[TEXT_COL].iloc[0][:200])

Документов: 194
Колонки   : ['path', 'filename', 'year', 'total_pages', 'text_pymupdf', 'text_body', 'clean_text', 'clean_wc', 'len_pymupdf', 'text_len', 'len_body', 'wc_pymupdf', 'wc_text', 'wc_body']

Пример clean_text (первые 200 симв):
дипломдық жобаны орындау кестесі ғылыми жетекші бөлімдер атаулары зерттелген лын мәселелер тізімі кеден ен ры мерз мдер пәндік саланы зертте 14 01 2019 ee қонышы жобалау бөлімі 28 02 2019 mol кітаптар


In [3]:
print(f'Загружаем {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model     = AutoModel.from_pretrained(MODEL_ID).to(device).eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Параметров: {n_params:.1f}M')
print(f'Hidden dim: {model.config.hidden_size}')
print(f'Max len   : {tokenizer.model_max_length}')

Загружаем ibm-granite/granite-embedding-311m-multilingual-r2...


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

Параметров: 311.7M
Hidden dim: 768
Max len   : 32768


In [ ]:
def chunk_document(text: str, tokenizer, max_len: int = MAX_LEN, overlap: int = OVERLAP):
    if not text or not text.strip():
        return []
    ids = tokenizer.encode(text, add_special_tokens=False)
    if not ids:
        return []

    stride = max_len - overlap
    window = max_len - 2
    step   = stride

    chunks = []
    for start in range(0, len(ids), step):
        window_ids = ids[start:start + window]
        if len(window_ids) < 16:
            break
        chunks.append({'token_start': start, 'token_end': start + len(window_ids), 'ids': window_ids})
        if start + window >= len(ids):
            break
    return chunks

all_chunks = []
for doc_idx, row in tqdm(df.iterrows(), total=len(df), desc='Chunking'):
    text = row[TEXT_COL]
    doc_chunks = chunk_document(text, tokenizer)
    for c_idx, ch in enumerate(doc_chunks):
        all_chunks.append({
            'doc_idx'    : doc_idx,
            'filename'   : row['filename'],
            'year'       : row['year'],
            'chunk_idx'  : c_idx,
            'token_start': ch['token_start'],
            'token_end'  : ch['token_end'],
            'ids'        : ch['ids'],
        })

chunks_df = pd.DataFrame(all_chunks)
n_chunks_per_doc = chunks_df.groupby('doc_idx').size()
print(f'Всего чанков      : {len(chunks_df)}')
print(f'Чанков на документ: min={n_chunks_per_doc.min()}, median={int(n_chunks_per_doc.median())}, max={n_chunks_per_doc.max()}, mean={n_chunks_per_doc.mean():.1f}')

Chunking:   0%|          | 0/194 [00:00<?, ?it/s]

Всего чанков      : 4986
Чанков на документ: min=2, median=24, max=69, mean=25.7


In [ ]:
def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


_CLS_ID = tokenizer.cls_token_id
_SEP_ID = tokenizer.sep_token_id
_PAD_ID = tokenizer.pad_token_id
if _PAD_ID is None:
    _PAD_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else 0
print(f'Спец-токены → CLS={_CLS_ID}, SEP={_SEP_ID}, PAD={_PAD_ID}')


def build_batch(chunk_ids_list, max_len=MAX_LEN):
    batch_ids = []
    for ids in chunk_ids_list:
        full = list(ids)
        if _CLS_ID is not None:
            full = [_CLS_ID] + full
        if _SEP_ID is not None:
            full = full + [_SEP_ID]
        full = full[:max_len]
        batch_ids.append(full)

    L = max(len(x) for x in batch_ids)
    input_ids = torch.full((len(batch_ids), L), _PAD_ID, dtype=torch.long)
    attention_mask = torch.zeros((len(batch_ids), L), dtype=torch.long)
    for i, ids in enumerate(batch_ids):
        n_tok = len(ids)
        input_ids[i, :n_tok] = torch.tensor(ids, dtype=torch.long)
        attention_mask[i, :n_tok] = 1
    return input_ids, attention_mask


@torch.no_grad()
def embed_chunks(chunks_df, model, batch_size=BATCH_SIZE, device=device):
    n = len(chunks_df)
    hidden = model.config.hidden_size
    out = np.zeros((n, hidden), dtype=np.float32)

    use_amp = device.type == 'cuda'

    for start in tqdm(range(0, n, batch_size), desc='Embedding'):
        end = min(start + batch_size, n)
        batch_ids_list = chunks_df['ids'].iloc[start:end].tolist()
        input_ids, attention_mask = build_batch(batch_ids_list)
        input_ids      = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)

        if use_amp:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                output = model(input_ids=input_ids, attention_mask=attention_mask)
                pooled = mean_pool(output.last_hidden_state, attention_mask)
                pooled = F.normalize(pooled, p=2, dim=1)
        else:
            output = model(input_ids=input_ids, attention_mask=attention_mask)
            pooled = mean_pool(output.last_hidden_state, attention_mask)
            pooled = F.normalize(pooled, p=2, dim=1)

        out[start:end] = pooled.float().cpu().numpy()

    return out

embeddings = embed_chunks(chunks_df, model)
print(f'\nЭмбеддинги: shape={embeddings.shape}, dtype={embeddings.dtype}')
print(f'Норма (должна быть ~1.0): mean={np.linalg.norm(embeddings, axis=1).mean():.4f}')

Спец-токены → CLS=None, SEP=None, PAD=0


Embedding:   0%|          | 0/312 [00:00<?, ?it/s]


Эмбеддинги: shape=(4986, 768), dtype=float32
Норма (должна быть ~1.0): mean=1.0000


In [ ]:
chunks_meta = chunks_df[['doc_idx', 'filename', 'year', 'chunk_idx', 'token_start', 'token_end']].copy()
chunks_meta.to_pickle(OUT_DIR / 'chunks_meta.pkl')

np.save(OUT_DIR / 'embeddings.npy', embeddings)

config = {
    'model_id'  : MODEL_ID,
    'max_len'   : MAX_LEN,
    'overlap'   : OVERLAP,
    'text_col'  : TEXT_COL,
    'hidden_dim': embeddings.shape[1],
    'n_docs'    : len(df),
    'n_chunks'  : len(chunks_meta),
}
with open(OUT_DIR / 'embeddings_config.pkl', 'wb') as f:
    pickle.dump(config, f)

print('Сохранено:')
print(f'  embeddings.npy       : {embeddings.nbytes / 1e6:.1f} MB  ({embeddings.shape})')
print(f'  chunks_meta.pkl      : {len(chunks_meta)} строк')
print(f'  embeddings_config.pkl: {config}')

Сохранено:
  embeddings.npy       : 15.3 MB  ((4986, 768))
  chunks_meta.pkl      : 4986 строк
  embeddings_config.pkl: {'model_id': 'ibm-granite/granite-embedding-311m-multilingual-r2', 'max_len': 512, 'overlap': 64, 'text_col': 'clean_text', 'hidden_dim': 768, 'n_docs': 194, 'n_chunks': 4986}


In [ ]:
rng = np.random.default_rng(42)

doc_counts = chunks_meta.groupby('doc_idx').size()
doc_a = doc_counts[doc_counts >= 3].index[0]
doc_b_candidates = doc_counts[(doc_counts >= 3) & (doc_counts.index != doc_a)].index
doc_b = rng.choice(doc_b_candidates)

idx_a = chunks_meta.index[chunks_meta['doc_idx'] == doc_a].tolist()
idx_b = chunks_meta.index[chunks_meta['doc_idx'] == doc_b].tolist()

emb_a = embeddings[idx_a]
emb_b = embeddings[idx_b]

intra = emb_a @ emb_a.T
np.fill_diagonal(intra, np.nan)
intra_mean = np.nanmean(intra)

inter_mean = (emb_a @ emb_b.T).mean()

print(f'Документ A: {df.loc[doc_a, "filename"][:70]}  ({len(idx_a)} чанков)')
print(f'Документ B: {df.loc[doc_b, "filename"][:70]}  ({len(idx_b)} чанков)')
print(f'\nCos sim внутри A       : {intra_mean:.3f}')
print(f'Cos sim A ↔ B (разные) : {inter_mean:.3f}')
print(f'\nGap                    : {intra_mean - inter_mean:.3f}  (должен быть > 0)')

Документ A: Tursynbek D.Кітаптарды онлайн жалға алуды ұйымдастыратын веб-сайт құру  (17 чанков)
Документ B: Жумабек Ж. Бет алпеты бойынша танып былу жуйесыне арналган камтама онд  (25 чанков)

Cos sim внутри A       : 0.669
Cos sim A ↔ B (разные) : 0.584

Gap                    : 0.084  (должен быть > 0)
